<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6D_Cell_6D_1C0_Final_Springer_Revision_Evidence_Package_Freeze_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# GES Stage 6D — Cell 6D-1C0
## Final Springer Revision Evidence & Reproducibility Package Freeze

**Purpose.** Freeze the scientific evidence package needed for the AI Therapeutics Editorial Board revision. This notebook creates **no new model and no new scientific experiment**. It packages and cross-checks the completed temporal validation, data-quality perturbation analysis, reference-resolution package, reproducibility provenance, and Editorial Comments 1–10 response map.

### Interpretation boundary
GES is treated as a **metadata-derived evidence stability index / relative evidence-uncertainty signal**, not a calibrated clinical probability and not patient-level predictive accuracy.



In [1]:

# Cell 1 — Setup, frozen paths, and hash helpers
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, platform, subprocess, sys
import pandas as pd
import numpy as np

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
STAGE4B_FEATURES = ROOT/'data_processed/stage4_ges/stage4b_t0_feature_transforms_and_weak_labels_v1.parquet'
STAGE4C_MODEL = ROOT/'models/stage4_ges/stage4c_full_ges_logistic_model_v1.joblib'
STAGE6B_COHORT = ROOT/'data_processed/stage6_temporal_validation/stage6b_locked_primary_evaluable_cohort_v1.parquet'
EXPECTED_SHA256 = {
 'stage4b_features':'c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8',
 'stage4c_full_ges_model':'0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30',
 'stage6b_evaluable_cohort':'c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038'}
A0_DIR=ROOT/'outputs/stage6_temporal_validation/stage6d_1a0_data_quality_perturbation_v1'
A0_MANIFEST=A0_DIR/'quality_checks/stage6d_1a0_output_manifest_v1.csv'
A0_SUMMARY=A0_DIR/'tables/stage6d_1a0_perturbation_summary_v1.csv'
A0_MANUSCRIPT_TABLE=A0_DIR/'tables/stage6d_1a0_manuscript_ready_robustness_table_v1.csv'
B1_DIR=ROOT/'outputs/revision_support/stage6d_1b1_reference_resolution_v1'
B1_MANIFEST=B1_DIR/'stage6d_1b1_manifest_v1.csv'
B1_RESOLUTION=B1_DIR/'stage6d_1b1_authoritative_reference_resolution_v1.csv'
B1_SUPPORT=B1_DIR/'stage6d_1b1_final_claim_support_map_v1.csv'
B1_CORRECTIONS=B1_DIR/'stage6d_1b1_bibliographic_correction_log_v1.csv'
B1_BIB=B1_DIR/'stage6d_1b1_corrected_bibliography_v1.txt'
OUTDIR=ROOT/'outputs/revision_support/stage6d_1c0_final_springer_evidence_package_v1'
OUTDIR.mkdir(parents=True, exist_ok=True)
def sha256_file(path, chunk_size=1024*1024):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  while True:
   b=f.read(chunk_size)
   if not b: break
   h.update(b)
 return h.hexdigest()
print('Output:',OUTDIR)


Mounted at /content/drive
Output: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1c0_final_springer_evidence_package_v1


In [2]:

# Cell 2 — Fail-closed verification of core frozen artifacts and completed reviewer packages
required={'stage4b_features':STAGE4B_FEATURES,'stage4c_full_ges_model':STAGE4C_MODEL,'stage6b_evaluable_cohort':STAGE6B_COHORT,
          'stage6d_1a0_manifest':A0_MANIFEST,'stage6d_1a0_summary':A0_SUMMARY,'stage6d_1a0_manuscript_table':A0_MANUSCRIPT_TABLE,
          'stage6d_1b1_manifest':B1_MANIFEST,'stage6d_1b1_resolution':B1_RESOLUTION,'stage6d_1b1_claim_support':B1_SUPPORT,
          'stage6d_1b1_corrections':B1_CORRECTIONS,'stage6d_1b1_bibliography':B1_BIB}
for name,p in required.items():
 if not p.exists(): raise FileNotFoundError(f'Missing {name}: {p}')
for name,p in {'stage4b_features':STAGE4B_FEATURES,'stage4c_full_ges_model':STAGE4C_MODEL,'stage6b_evaluable_cohort':STAGE6B_COHORT}.items():
 obs=sha256_file(p); exp=EXPECTED_SHA256[name]
 print(name,obs)
 if obs!=exp: raise RuntimeError(f'Frozen SHA-256 mismatch for {name}')
a0_manifest=pd.read_csv(A0_MANIFEST); b1_manifest=pd.read_csv(B1_MANIFEST)
if len(a0_manifest)<7 or len(b1_manifest)<5: raise RuntimeError('Prior package manifest unexpectedly incomplete.')
print('PASS — required frozen sources found.')


stage4b_features c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8
stage4c_full_ges_model 0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30
stage6b_evaluable_cohort c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038
PASS — required frozen sources found.


In [3]:

# Cell 3 — Verify prior package manifests have not changed
def verify_manifest(df):
 rows=[]
 for _,r in df.iterrows():
  p=Path(str(r['artifact'])); exp=str(r['sha256'])
  if not p.exists(): rows.append((str(p),'MISSING','',exp)); continue
  obs=sha256_file(p); rows.append((str(p),'PASS' if obs==exp else 'HASH_MISMATCH',obs,exp))
 return pd.DataFrame(rows,columns=['artifact','status','observed_sha256','expected_sha256'])
a0_check=verify_manifest(a0_manifest); b1_check=verify_manifest(b1_manifest)
display(a0_check); display(b1_check)
if not (a0_check.status=='PASS').all(): raise RuntimeError('1A0 package changed after freeze.')
if not (b1_check.status=='PASS').all(): raise RuntimeError('1B1 package changed after freeze.')
print('PASS — 1A0 and 1B1 hashes unchanged.')


,artifact,status,observed_sha256,expected_sha256
0,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,13f23ba6c88ae63d5d788ef375cc38ef3a344a6199707f...,13f23ba6c88ae63d5d788ef375cc38ef3a344a6199707f...
1,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,4c1a055adafb8da3184353895e4e0700833a3ff5efd39b...,4c1a055adafb8da3184353895e4e0700833a3ff5efd39b...
2,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,08a863e8fd813226d2c2f4623f0f797f31184f3db250e3...,08a863e8fd813226d2c2f4623f0f797f31184f3db250e3...
3,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,c272d5b4061ce8a670a6d0bd0cd434762b60fe6f76183c...,c272d5b4061ce8a670a6d0bd0cd434762b60fe6f76183c...
4,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,30566428dcd02b1e642eb4431df2e2c8b1e647517cf735...,30566428dcd02b1e642eb4431df2e2c8b1e647517cf735...
5,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,ff1630a64730f1c73d9ea371f25072e5db71abea775825...,ff1630a64730f1c73d9ea371f25072e5db71abea775825...
6,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,9ef4483a08961557d42bbc8d0e8028fe9d38d2617da989...,9ef4483a08961557d42bbc8d0e8028fe9d38d2617da989...


,artifact,status,observed_sha256,expected_sha256
0,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,44576af9b858455854d9c5e78a723f1ade657ce02f4797...,44576af9b858455854d9c5e78a723f1ade657ce02f4797...
1,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,642460ce45379f733261a0d7bf1c470aa23edd1812bdf7...,642460ce45379f733261a0d7bf1c470aa23edd1812bdf7...
2,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,4c058d03b0f982b33fb89eae6ec669cdc0e654c0a941cf...,4c058d03b0f982b33fb89eae6ec669cdc0e654c0a941cf...
3,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,0ee1c36da193a734d99cae8b181b78be7c7e803ac9e1a0...,0ee1c36da193a734d99cae8b181b78be7c7e803ac9e1a0...
4,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,PASS,0be6c3b56bc878183e7c4c216ad354e045d3ce711f74a5...,0be6c3b56bc878183e7c4c216ad354e045d3ce711f74a5...


PASS — 1A0 and 1B1 hashes unchanged.


In [4]:

# Cell 4 — Cross-check locked temporal cohort and freeze study-source ledger
cohort=pd.read_parquet(STAGE6B_COHORT)
if len(cohort)!=66636: raise RuntimeError(f'Expected 66,636 evaluable RCVs; found {len(cohort):,}')
OUTCOME_COL='primary_future_instability'
if OUTCOME_COL not in cohort.columns: raise KeyError(OUTCOME_COL)
events=int(cohort[OUTCOME_COL].astype(int).sum()); negatives=len(cohort)-events; prevalence=events/len(cohort)
if events!=6485: raise RuntimeError(f'Expected 6,485 events; found {events:,}')
source_ledger=pd.DataFrame([
 {'component':'T0 ClinVar RCV archive','frozen_description':'January 2023 ClinVar RCV archive','embedded_cutoff':'2022-12-31','record_count':71659,'role':'T0-only GES reconstruction and baseline metadata'},
 {'component':'T1 ClinVar RCV archive','frozen_description':'January 2026 ClinVar RCV archive','embedded_cutoff':'2025-12-27','record_count':100920,'role':'Future outcome and later evidence corpus'},
 {'component':'Locked temporal evaluable cohort','frozen_description':'RCV-level linked/evaluable T0→T1 cohort','embedded_cutoff':'T0→T1','record_count':len(cohort),'role':f'Independent temporal evaluation; {events} events; {negatives} negatives'}])
source_ledger['target_genes']='BRCA1; BRCA2; MLH1; EGFR'
source_ledger['clinical_boundary']='Knowledge-record / public metadata study; no patient-level outcome'
source_ledger_path=OUTDIR/'stage6d_1c0_study_source_ledger_v1.csv'; source_ledger.to_csv(source_ledger_path,index=False)
print('n=',len(cohort),'events=',events,'prevalence=',prevalence); display(source_ledger)


n= 66636 events= 6485 prevalence= 0.09731976709286272


,component,frozen_description,embedded_cutoff,record_count,role,target_genes,clinical_boundary
0,T0 ClinVar RCV archive,January 2023 ClinVar RCV archive,2022-12-31,71659,T0-only GES reconstruction and baseline metadata,BRCA1; BRCA2; MLH1; EGFR,Knowledge-record / public metadata study; no p...
1,T1 ClinVar RCV archive,January 2026 ClinVar RCV archive,2025-12-27,100920,Future outcome and later evidence corpus,BRCA1; BRCA2; MLH1; EGFR,Knowledge-record / public metadata study; no p...
2,Locked temporal evaluable cohort,RCV-level linked/evaluable T0→T1 cohort,T0→T1,66636,Independent temporal evaluation; 6485 events; ...,BRCA1; BRCA2; MLH1; EGFR,Knowledge-record / public metadata study; no p...


In [5]:

# Cell 5 — Freeze quantitative evidence summary
A=pd.read_csv(A0_SUMMARY); R=pd.read_csv(B1_RESOLUTION).fillna(''); S=pd.read_csv(B1_SUPPORT).fillna('')
if A.scenario_id.nunique()!=15: raise RuntimeError('Expected 15 perturbation scenarios.')
if len(R)!=17 or not (R.claim_support=='SUPPORTED').all(): raise RuntimeError('Reference-resolution gate failed.')
worst=A.loc[A.delta_auprc_median.idxmin()]; best=A.loc[A.delta_auprc_median.idxmax()]
evidence_summary=pd.DataFrame([
 ['TEMP-001','Temporal evaluable RCVs',66636,'Independent T0→T1 cohort'],
 ['TEMP-002','Future-instability events',6485,'Frozen future outcome'],
 ['TEMP-003','Future-instability prevalence',prevalence,'Knowledge-record event prevalence'],
 ['TEMP-004','Full-GES AUPRC',0.11244406235413203,'Weak temporal ranking signal'],
 ['TEMP-005','Full-GES AUROC',0.5358121831324858,'Weak absolute discrimination'],
 ['ROB-001','Data-quality perturbation scenarios',int(A.scenario_id.nunique()),'Revision-motivated robustness analysis'],
 ['ROB-002','Worst median AUPRC delta',float(worst.delta_auprc_median),str(worst.scenario_id)],
 ['ROB-003','Largest positive median AUPRC delta',float(best.delta_auprc_median),str(best.scenario_id)],
 ['REF-001','Resolved retained references',len(R),'Authoritative resolution'],
 ['REF-002','Bibliographic corrections',int((R.decision=='correct').sum()),'Corrected bibliography'],
 ['REF-003','Erratum-reviewed retained sources',int((R.decision=='retain_with_erratum_note').sum()),'Erratum reviewed']
],columns=['evidence_id','measure','value','interpretation'])
evidence_summary_path=OUTDIR/'stage6d_1c0_quantitative_evidence_summary_v1.csv'; evidence_summary.to_csv(evidence_summary_path,index=False)
display(evidence_summary)


,evidence_id,measure,value,interpretation
0,TEMP-001,Temporal evaluable RCVs,66636.000000,Independent T0→T1 cohort
1,TEMP-002,Future-instability events,6485.000000,Frozen future outcome
2,TEMP-003,Future-instability prevalence,0.097320,Knowledge-record event prevalence
3,TEMP-004,Full-GES AUPRC,0.112444,Weak temporal ranking signal
4,TEMP-005,Full-GES AUROC,0.535812,Weak absolute discrimination
5,ROB-001,Data-quality perturbation scenarios,15.000000,Revision-motivated robustness analysis
6,ROB-002,Worst median AUPRC delta,-0.006456,conflict_false_positive_10
7,ROB-003,Largest positive median AUPRC delta,0.009183,recency_stale_plus_2y_all
8,REF-001,Resolved retained references,17.000000,Authoritative resolution
9,REF-002,Bibliographic corrections,3.000000,Corrected bibliography


In [6]:

# Cell 6 — Required terminology boundaries
terminology=pd.DataFrame([
 ['clinical probability / probability evidence is correct','metadata-derived evidence stability index / relative evidence-uncertainty signal','Poor temporal calibration; not a validated clinical probability'],
 ['AUC=1.0 proves predictive validity','AUC=1.0 is an internal proxy-label reconstruction / implementation check only','Labels use the same metadata families as predictors'],
 ['High Risk / Low Risk as validated clinical classes','operational GES score strata / relative prioritization','Thresholds were not clinically calibrated'],
 ['GES predicts therapeutic or patient outcome','GES ranks metadata-level evidence fragility / future ClinVar knowledge-record instability','No patient outcome was evaluated'],
 ['GES is a universally superior RAG reranker','No clear primary composite benefit; caution-policy secondary effect observed','Primary composite interval included zero']
],columns=['avoid','preferred','reason'])
terminology_path=OUTDIR/'stage6d_1c0_required_terminology_boundaries_v1.csv'; terminology.to_csv(terminology_path,index=False); display(terminology)


,avoid,preferred,reason
0,clinical probability / probability evidence is...,metadata-derived evidence stability index / re...,Poor temporal calibration; not a validated cli...
1,AUC=1.0 proves predictive validity,AUC=1.0 is an internal proxy-label reconstruct...,Labels use the same metadata families as predi...
2,High Risk / Low Risk as validated clinical cla...,operational GES score strata / relative priori...,Thresholds were not clinically calibrated
3,GES predicts therapeutic or patient outcome,GES ranks metadata-level evidence fragility / ...,No patient outcome was evaluated
4,GES is a universally superior RAG reranker,No clear primary composite benefit; caution-po...,Primary composite interval included zero


In [7]:

# Cell 7 — Editorial Comments 1–10 evidence map
rows=[
 [1,'Interpretation / circular proxy-label issue','SCIENTIFICALLY_ADDRESSED','Temporal validation + circularity disclosure','State prominently in abstract/methods/results/discussion/conclusion that proxy-label AUC=1.0 is structurally expected and not external predictive validity.'],
 [2,'Terminology','SCIENTIFICALLY_ADDRESSED',str(terminology_path),'Replace clinical-probability language with metadata-derived evidence index / score terminology throughout.'],
 [3,'Independent validation + uncertainty + perturbations','SCIENTIFICALLY_ADDRESSED',f'{STAGE6B_COHORT}; {A0_MANUSCRIPT_TABLE}','Add locked Jan-2023→Jan-2026 validation, bootstrap/sensitivity results, and 1A0 metadata perturbation robustness.'],
 [4,'Reproducibility','PACKAGE_ADDRESSED','Frozen hashes + source ledger + repository commit + manifests','Report exact snapshot dates/cutoffs, counts, commit, software environment, and permanent code/data statement; add Zenodo DOI if created.'],
 [5,'Clinical boundary','SCIENTIFICALLY_ADDRESSED',str(terminology_path),'State no patient-level clinical validation; list evidence needed before therapeutic decision support.'],
 [6,'Figures / charts / copyright','FINAL_MANUSCRIPT_ACTION_REQUIRED',str(A0_DIR/'figures'),'Retain author-created plots/diagrams only; provide plotting data, editable sources, captions, and rights records.'],
 [7,'Reference verification','ADDRESSED',f'{B1_RESOLUTION}; {B1_SUPPORT}; {B1_CORRECTIONS}','Use corrected 1B1 bibliography and keep verification record.'],
 [8,'Authorship / originality / generative-AI disclosure','AUTHOR_CONFIRMATION_REQUIRED','1C0 author checklist','Provide truthful signed declaration covering every actual generative-AI use; distinguish experimental LLM use from writing/coding assistance.'],
 [9,'Manuscript information / declarations','AUTHOR_CONFIRMATION_REQUIRED','1C0 author checklist','Confirm title, abstract, keywords, author/affiliation/country, active corresponding email, COI, funding, contributions, data/code, ethics/consent, and AI-use declarations.'],
 [10,'Revision package / response letter','PACKAGE_TEMPLATE_READY','1C0 response skeleton + manifest','Prepare clean DOCX/PDF, numbered response with final page/section references, editable figures/data/permissions and code/data links.']]
editorial_map=pd.DataFrame(rows,columns=['comment','topic','status','evidence','required_revision'])
editorial_map_path=OUTDIR/'stage6d_1c0_editorial_comments_1_to_10_evidence_map_v1.csv'; editorial_map.to_csv(editorial_map_path,index=False); display(editorial_map)


,comment,topic,status,evidence,required_revision
0,1,Interpretation / circular proxy-label issue,SCIENTIFICALLY_ADDRESSED,Temporal validation + circularity disclosure,State prominently in abstract/methods/results/...
1,2,Terminology,SCIENTIFICALLY_ADDRESSED,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,Replace clinical-probability language with met...
2,3,Independent validation + uncertainty + perturb...,SCIENTIFICALLY_ADDRESSED,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,"Add locked Jan-2023→Jan-2026 validation, boots..."
3,4,Reproducibility,PACKAGE_ADDRESSED,Frozen hashes + source ledger + repository com...,"Report exact snapshot dates/cutoffs, counts, c..."
4,5,Clinical boundary,SCIENTIFICALLY_ADDRESSED,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,State no patient-level clinical validation; li...
5,6,Figures / charts / copyright,FINAL_MANUSCRIPT_ACTION_REQUIRED,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,Retain author-created plots/diagrams only; pro...
6,7,Reference verification,ADDRESSED,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,Use corrected 1B1 bibliography and keep verifi...
7,8,Authorship / originality / generative-AI discl...,AUTHOR_CONFIRMATION_REQUIRED,1C0 author checklist,Provide truthful signed declaration covering e...
8,9,Manuscript information / declarations,AUTHOR_CONFIRMATION_REQUIRED,1C0 author checklist,"Confirm title, abstract, keywords, author/affi..."
9,10,Revision package / response letter,PACKAGE_TEMPLATE_READY,1C0 response skeleton + manifest,"Prepare clean DOCX/PDF, numbered response with..."


In [8]:

# Cell 8 — Author-confirmation checklist
check=pd.DataFrame([
 ['Final chapter title','CONFIRM','Suggested: Quantifying Genomic Evidence Reliability for Trustworthy Biomedical AI in Precision Oncology: A Metadata-Derived Evidence Index with Temporal Validation'],
 ['Corresponding-author email','CONFIRM_ACTIVE','Use a working email address; do not submit an inactive institutional address.'],
 ['Funding','CONFIRM','State actual funding or explicitly no external funding, if true.'],
 ['Competing interests','CONFIRM','Declare actual competing interests or none, if true.'],
 ['Author contribution','CONFIRM','Verify the contribution statement against actual work.'],
 ['Ethics / consent','CONFIRM','Public ClinVar data; confirm publisher wording.'],
 ['Generative-AI drafting/editing/coding/analysis/images','CONFIRM_FULL_DISCLOSURE','Disclose every actual use; do not under-report.'],
 ['Experimental LLM use','CONFIRM','Retain GPT-4.1 mini experimental-model disclosure if RAG experiment remains in chapter.'],
 ['Data availability','CONFIRM','Public ClinVar archives + frozen snapshot/cutoff details.'],
 ['Code availability','CONFIRM','GitHub frozen commit; add Zenodo DOI if archived.'],
 ['Figure rights','CONFIRM','All figures author-created or permissions/license documented.']
],columns=['item','status','suggested_text'])
author_checklist_path=OUTDIR/'stage6d_1c0_author_confirmation_checklist_v1.csv'; check.to_csv(author_checklist_path,index=False); display(check)


,item,status,suggested_text
0,Final chapter title,CONFIRM,Suggested: Quantifying Genomic Evidence Reliab...
1,Corresponding-author email,CONFIRM_ACTIVE,Use a working email address; do not submit an ...
2,Funding,CONFIRM,State actual funding or explicitly no external...
3,Competing interests,CONFIRM,"Declare actual competing interests or none, if..."
4,Author contribution,CONFIRM,Verify the contribution statement against actu...
5,Ethics / consent,CONFIRM,Public ClinVar data; confirm publisher wording.
6,Generative-AI drafting/editing/coding/analysis...,CONFIRM_FULL_DISCLOSURE,Disclose every actual use; do not under-report.
7,Experimental LLM use,CONFIRM,Retain GPT-4.1 mini experimental-model disclos...
8,Data availability,CONFIRM,Public ClinVar archives + frozen snapshot/cuto...
9,Code availability,CONFIRM,GitHub frozen commit; add Zenodo DOI if archived.


In [9]:

# Cell 9 — Repository commit and provenance record
REPO_URL='https://github.com/SANGHATI23/genomic-evidence-reliability.git'
def remote_head(url):
 try:
  out=subprocess.check_output(['git','ls-remote',url,'HEAD'],text=True,timeout=30).strip(); return out.split()[0] if out else ''
 except Exception as e: return f'UNRESOLVED: {repr(e)}'
git_head=remote_head(REPO_URL)
prov={'package_id':'GES_STAGE6D_1C0_FINAL_SPRINGER_REVISION_EVIDENCE_PACKAGE_V1','created_utc':pd.Timestamp.utcnow().isoformat(),
      'repository':REPO_URL,'remote_git_head_at_package_freeze':git_head,'frozen_core_sha256':EXPECTED_SHA256,
      'stage6d_1a0_manifest_sha256':sha256_file(A0_MANIFEST),'stage6d_1b1_manifest_sha256':sha256_file(B1_MANIFEST),
      'environment':{'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__},
      'interpretation_boundary':'Metadata-derived evidence index; not calibrated clinical probability, patient-level predictive accuracy, therapeutic efficacy, or clinical safety.'}
provenance_path=OUTDIR/'stage6d_1c0_package_provenance_v1.json'; provenance_path.write_text(json.dumps(prov,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print('Repository HEAD:',git_head); print('Provenance:',provenance_path)


Repository HEAD: 9deca0e2c94bfef93a914556de73da7e79af9570
Provenance: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1c0_final_springer_evidence_package_v1/stage6d_1c0_package_provenance_v1.json


In [10]:

# Cell 10 — Point-by-point response skeleton
lines=['AI Therapeutics Book Project — Point-by-Point Revision Response Skeleton','',
       'Chapter: Quantifying Genomic Evidence Reliability for Trustworthy Biomedical AI in Precision Oncology','']
for _,r in editorial_map.sort_values('comment').iterrows():
 lines += [f"COMMENT {int(r.comment)}: {r.topic}",'','EDITORIAL COMMENT:','[Paste the exact Editorial Board comment here.]','',
           'RESPONSE:',f'[Describe the revision. Evidence status: {r.status}.]','',
           'CHANGES MADE:',r.required_revision,'','LOCATION IN REVISED MANUSCRIPT:','[Insert final page and section after DOCX/PDF is finalized.]','',
           'SUPPORTING EVIDENCE:',str(r.evidence),'','-'*80,'']
response_path=OUTDIR/'stage6d_1c0_point_by_point_response_skeleton_v1.txt'; response_path.write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('Response skeleton:',response_path)


Response skeleton: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1c0_final_springer_evidence_package_v1/stage6d_1c0_point_by_point_response_skeleton_v1.txt


In [11]:

# Cell 11 — Manuscript edit map
edit_map=pd.DataFrame([
 ['Title','REMOVE_PROBABILITY_CLAIM','Use metadata-derived evidence index / temporal-validation framing.'],
 ['Abstract','MAJOR_REWRITE','State circularity; summarize independent temporal results; weak ranking / poor calibration; robustness; non-clinical boundary.'],
 ['Introduction','UPDATE','Frame knowledge-source uncertainty and distinguish metadata reliability from model uncertainty.'],
 ['Methods — GES derivation','UPDATE','Describe weak labels transparently; in-sample AUC is only an implementation/internal-consistency check.'],
 ['Methods — temporal validation','ADD','January 2023 T0-only reconstruction; January 2026 future outcome; RCV linkage; comparators; AUPRC/AUROC; bootstrap; calibration.'],
 ['Methods — data-quality perturbations','ADD','Frozen-model 1A0 protocol; no retraining or T1-informed repair.'],
 ['Results — cross-sectional','DOWNWEIGHT','Do not present AUC=1.0 as predictive validity.'],
 ['Results — temporal validation','ADD_CENTERPIECE','n=66,636; 6,485 events; Full-GES AUPRC≈0.11244; AUROC≈0.53581; weak signal; poor calibration; comparator/gene sensitivity.'],
 ['Results — perturbation robustness','ADD',f"15 scenarios; worst median AUPRC delta {float(worst.delta_auprc_median):+.6f} ({worst.scenario_id}); heterogeneous effects."],
 ['Discussion','MAJOR_REWRITE','Interpret narrowly as evidence-uncertainty/prioritization signal; discuss simple metadata baseline strength and transport limits.'],
 ['Clinical boundary','STRENGTHEN','No patient outcomes/safety; list evidence needed before therapeutic decisions.'],
 ['Limitations','EXPAND','Weak-label origin, modest discrimination, poor calibration, gene heterogeneity, perturbation heterogeneity, no prospective clinical validation.'],
 ['Conclusion','MAJOR_REWRITE','No clinical probability claim; emphasize temporal evidence-uncertainty signal and need for expert/prospective validation.'],
 ['References','REPLACE_WITH_1B1','Use corrected 17-reference bibliography.'],
 ['Declarations','FINALIZE','Funding, COI, contributions, ethics/consent, AI disclosure, data/code, active email.']
],columns=['section','action','content'])
edit_map_path=OUTDIR/'stage6d_1c0_manuscript_edit_map_v1.csv'; edit_map.to_csv(edit_map_path,index=False); display(edit_map)


,section,action,content
0,Title,REMOVE_PROBABILITY_CLAIM,Use metadata-derived evidence index / temporal...
1,Abstract,MAJOR_REWRITE,State circularity; summarize independent tempo...
2,Introduction,UPDATE,Frame knowledge-source uncertainty and disting...
3,Methods — GES derivation,UPDATE,Describe weak labels transparently; in-sample ...
4,Methods — temporal validation,ADD,January 2023 T0-only reconstruction; January 2...
5,Methods — data-quality perturbations,ADD,Frozen-model 1A0 protocol; no retraining or T1...
6,Results — cross-sectional,DOWNWEIGHT,Do not present AUC=1.0 as predictive validity.
7,Results — temporal validation,ADD_CENTERPIECE,"n=66,636; 6,485 events; Full-GES AUPRC≈0.11244..."
8,Results — perturbation robustness,ADD,15 scenarios; worst median AUPRC delta -0.0064...
9,Discussion,MAJOR_REWRITE,Interpret narrowly as evidence-uncertainty/pri...


In [12]:

# Cell 12 — Freeze final package manifest and PASS gate
package_artifacts=[source_ledger_path,evidence_summary_path,terminology_path,editorial_map_path,author_checklist_path,provenance_path,response_path,edit_map_path,
                   A0_MANUSCRIPT_TABLE,B1_RESOLUTION,B1_SUPPORT,B1_CORRECTIONS,B1_BIB]
rows=[]
for p in package_artifacts:
 p=Path(p)
 if not p.exists(): raise FileNotFoundError(p)
 rows.append({'artifact':str(p),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
manifest=pd.DataFrame(rows); manifest_path=OUTDIR/'stage6d_1c0_final_evidence_package_manifest_v1.csv'; manifest.to_csv(manifest_path,index=False)
for p in package_artifacts+[manifest_path]:
 p=Path(p); p.with_name(p.name+'.sha256').write_text(f'{sha256_file(p)}  {p.name}\n',encoding='utf-8')
ready=(len(cohort)==66636 and events==6485 and A.scenario_id.nunique()==15 and len(R)==17 and (R.claim_support=='SUPPORTED').all() and (a0_check.status=='PASS').all() and (b1_check.status=='PASS').all())
print('FINAL SPRINGER REVISION EVIDENCE PACKAGE'); print('----------------------------------------')
print('Temporal evaluable RCVs:',len(cohort)); print('Future-instability events:',events); print('Perturbation scenarios:',A.scenario_id.nunique()); print('Resolved references:',len(R)); print('Manifest:',manifest_path); print('Repository HEAD:',git_head)
if not ready: raise RuntimeError('Scientific evidence package did not pass all readiness gates.')
print('\nPASS — Stage 6D-1C0 scientific evidence package frozen.')
print('STATUS: READY FOR REVISED CHAPTER + POINT-BY-POINT RESPONSE DRAFTING.')
print('NO NEW SCIENTIFIC EXPERIMENT IS REQUIRED BY THIS PACKAGE.')
print('AUTHOR CONFIRMATION STILL REQUIRED for declarations, active email, AI-use disclosure, and final figure rights.')


FINAL SPRINGER REVISION EVIDENCE PACKAGE
----------------------------------------
Temporal evaluable RCVs: 66636
Future-instability events: 6485
Perturbation scenarios: 15
Resolved references: 17
Manifest: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1c0_final_springer_evidence_package_v1/stage6d_1c0_final_evidence_package_manifest_v1.csv
Repository HEAD: 9deca0e2c94bfef93a914556de73da7e79af9570

PASS — Stage 6D-1C0 scientific evidence package frozen.
STATUS: READY FOR REVISED CHAPTER + POINT-BY-POINT RESPONSE DRAFTING.
NO NEW SCIENTIFIC EXPERIMENT IS REQUIRED BY THIS PACKAGE.
AUTHOR CONFIRMATION STILL REQUIRED for declarations, active email, AI-use disclosure, and final figure rights.
